In [12]:
# Importation des packages

In [11]:
import kagglehub
import os
import pandas as pd

# Télécharger le dataset
path = kagglehub.dataset_download("mirichoi0218/insurance")
print("Chemin du dossier :", path)

# Lister les fichiers
files = os.listdir(path)
print("Fichiers disponibles :", files)

# Charger le CSV
csv_path = os.path.join(path, "insurance.csv")
df = pd.read_csv(csv_path)

# Afficher les 10 premières lignes
display(df.head(10))


Using Colab cache for faster access to the 'insurance' dataset.
Chemin du dossier : /kaggle/input/insurance
Fichiers disponibles : ['insurance.csv']


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520
5,31,female,25.740,0,no,southeast,3756.62160
6,46,female,33.440,1,no,southeast,8240.58960
7,37,female,27.740,3,no,northwest,7281.50560
8,37,male,29.830,2,no,northeast,6406.41070
9,60,female,25.840,0,no,northwest,28923.13692


Sélection de la variable cible et des variables explicatives

In [16]:
# 1) On supprime les lignes contenant des valeurs manquantes
# (on garde uniquement les observations complètes)
df = df.dropna()

# 2) Choix de la variable cible (y)
# Ici, la variable à prédire est "charges" (coût des soins médicaux)
y = df["charges"]   # y est une série (vector) cible

# 3) Choix des variables explicatives (X)
# On enlève la colonne cible "charges" des variables explicatives
X = df.drop(columns=["charges"])

# 4) Affichage des choix effectués
print("\n Variable cible (y) :", y.name)
print("\n Variables explicatives (X) :", list(X.columns))

# Option : enlever la variable 'region' si elle n'est pas importante

# Si tu décides de retirer 'region' (ex : faible contribution / pas utile),
# alors tu peux la supprimer de X (sans toucher à y)
if "region" in X.columns:
    X_sans_region = X.drop(columns=["region"])
    print("\n Variables explicatives SANS 'region' :", list(X_sans_region.columns))
else:
    X_sans_region = X
    print("\nℹ La variable 'region' n'existe pas dans X, rien à supprimer.")



 Variable cible (y) : charges

 Variables explicatives (X) : ['age', 'sex', 'bmi', 'children', 'smoker', 'region']

 Variables explicatives SANS 'region' : ['age', 'sex', 'bmi', 'children', 'smoker']


**Justification des choix des modèles linéaires et non linéaires et des metrics**

**Metric**

**RMSE (Root Mean Squared Error)**

Le RMSE mesure l’erreur moyenne en pénalisant fortement les grandes erreurs de prédiction, ce qui est particulièrement pertinent pour les coûts médicaux élevés. Il s’exprime dans la même unité que la variable cible, facilitant l’interprétation. En revanche, il est sensible aux valeurs extrêmes et peut être dominé par quelques observations atypiques.

**MAE (Mean Absolute Error)**

Le **MAE** fournit une mesure simple et robuste de l’erreur moyenne, moins influencée par les valeurs extrêmes que le RMSE. Il est facile à interpréter et stable lorsque les coûts sont très dispersés. Toutefois, il pénalise moins fortement les grandes erreurs, ce qui peut être limitant dans un contexte assurantiel.

**R² (Coefficient de détermination)**

Le coefficient **R²** indique la proportion de la variabilité des coûts médicaux expliquée par le modèle, ce qui en fait un indicateur global de qualité d’ajustement. Il est particulièrement utile pour comparer plusieurs modèles. Cependant, il ne renseigne pas directement sur l’ampleur réelle des erreurs de prédiction.

**MSE (Mean Squared Error)**

Le **MSE** est simple à calculer et largement utilisé dans l’optimisation des modèles linéaires. Il pénalise fortement les grandes erreurs, ce qui peut être utile théoriquement. Néanmoins, son unité au carré rend l’interprétation difficile et il est très sensible aux valeurs extrêmes.

**MAPE (Mean Absolute Percentage Error)**

Le **MAPE** exprime l’erreur en pourcentage, ce qui peut sembler intuitif. Toutefois, il devient instable lorsque la variable cible prend de faibles valeurs et est peu adapté à des coûts médicaux hétérogènes. Pour cette raison, il n’est pas retenu dans ce projet.

### **Modèle linéaire**

### **Régression linéaire**

La régression linéaire est simple à estimer et très interprétable, ce qui permet de mesurer l’effet marginal de chaque variable explicative sur les coûts médicaux. Elle constitue un bon modèle de référence. En revanche, elle suppose une relation linéaire et peut mal capturer des interactions complexes.

---

### **Ridge (régression pénalisée L2)**

La régression Ridge réduit le sur-apprentissage en pénalisant la taille des coefficients, ce qui améliore la stabilité du modèle en présence de multicolinéarité. Elle conserve toutes les variables explicatives. Toutefois, elle n’effectue pas de sélection automatique des variables.

---

### **Lasso (régression pénalisée L1)**

La régression Lasso permet à la fois la régularisation et la sélection de variables en annulant certains coefficients. Elle est utile pour simplifier le modèle et améliorer l’interprétabilité. Cependant, elle peut être instable lorsque les variables sont fortement corrélées.

---

### **Elastic Net**

L’Elastic Net combine les pénalités Ridge et Lasso, offrant un bon compromis entre stabilité et sélection de variables. Il est bien adapté aux données présentant de la multicolinéarité. En contrepartie, il nécessite le réglage de plusieurs hyperparamètres.


**Modèles non linéaires**

**KNN (k-plus proches voisins)**

Le **KNN** permet de modéliser des relations non linéaires sans faire d’hypothèse sur la forme du modèle. Il est simple et intuitif pour comparer des profils similaires d’assurés. Toutefois, il est sensible à l’échelle des variables et devient coûteux lorsque la taille des données augmente.

**Arbre de décision**

L’**arbre de décision** est facilement interprétable et capte naturellement les interactions entre variables explicatives. Il est utile pour comprendre les mécanismes de décision. En revanche, il est très sensible au sur-apprentissage s’il n’est pas régularisé.

**Random Forest**

Le **Random Forest** améliore la stabilité et la précision des arbres de décision en réduisant la variance. Il est particulièrement performant sur des données tabulaires comme celles des coûts médicaux. Cependant, son interprétation est moins directe et son coût de calcul est plus élevé.

**Bagging**

Le **Bagging** permet de réduire la variabilité des prédictions en combinant plusieurs modèles entraînés sur des échantillons différents. Il améliore la robustesse par rapport à un arbre unique. Toutefois, il ne réduit pas le biais si le modèle de base est trop simple.

**Boosting**

Les méthodes de **boosting** offrent une très forte capacité de prédiction en corrigeant progressivement les erreurs des modèles précédents. Elles sont particulièrement adaptées à des relations complexes entre variables. En revanche, elles nécessitent un réglage fin des hyperparamètres pour éviter le sur-apprentissage.

**SVM (Machines à vecteurs de support)**

Les **SVM** permettent de modéliser des relations non linéaires grâce aux fonctions noyau et offrent une bonne généralisation. Elles sont efficaces sur des jeux de données de taille modérée. Cependant, elles sont peu interprétables et sensibles au choix des hyperparamètres.

**Réseaux de neurones**

Les **réseaux de neurones** possèdent une grande capacité de modélisation des relations non linéaires complexes. Néanmoins, ils requièrent un volume important de données et un réglage complexe. Sur des données tabulaires de taille modérée, leur avantage n’est pas garanti.

**Conclusion sur l’évaluation des modèles.**

Dans ce projet de prédiction des coûts médicaux, les modèles linéaires et non linéaires sont évalués à l’aide des métriques **RMSE**, **MAE** et du coefficient de détermination **R²**. Le **RMSE** est privilégié comme métrique principale car il pénalise fortement les erreurs importantes, particulièrement critiques dans un contexte assurantiel. Le **MAE** est utilisé en complément pour apprécier la robustesse des modèles face aux valeurs extrêmes. Le **R²** permet d’évaluer la qualité globale de l’ajustement et de comparer les performances entre modèles. Enfin, pour les **modèles non linéaires**, une attention particulière est portée à la comparaison des performances entre les données d’entraînement et de test afin de détecter d’éventuels phénomènes de sur-apprentissage.